# 01 — Data Audit

## Objective

Validate the structure, completeness, consistency, and integrity of the
NovoStar datasets before conducting marketing effectiveness analysis.

The audit covers:

- dataset structure and data types
- missing values
- duplicate records
- primary and foreign-key integrity
- date ranges and date consistency
- campaign attribution
- order status and revenue
- order/order-item reconciliation
- product and customer sanity checks
- support-ticket quality

The results of this audit determine which analytical assumptions and
business metrics can be used in the subsequent analysis.

In [1]:
import pandas as pd
import numpy as np

from IPython.display import display

from src.data_loader import load_data
from src.data_quality import (
    missing_summary,
    duplicate_count,
    foreign_key_check,
    date_range_summary,
)

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

data = load_data()

customers = data["customers"]
products = data["products"]
orders = data["orders"]
order_items = data["order_items"]
campaigns = data["campaigns"]
support = data["support"]

print("Data loaded successfully.")

Data loaded successfully.


## 1. Dataset overview

First, inspect the size and basic structure of each dataset.

In [2]:
datasets = {
    "customers": customers,
    "products": products,
    "orders": orders,
    "order_items": order_items,
    "campaigns": campaigns,
    "support_tickets": support,
}

overview = pd.DataFrame(
    {
        "rows": {name: df.shape[0] for name, df in datasets.items()},
        "columns": {name: df.shape[1] for name, df in datasets.items()},
        "duplicate_rows": {
            name: df.duplicated().sum()
            for name, df in datasets.items()
        },
    }
)

display(overview)

,rows,columns,duplicate_rows
customers,3157,10,37
products,150,9,0
orders,15164,11,224
order_items,29887,5,97
campaigns,12,6,0
support_tickets,1540,9,0


## 2. Schema and data types

Check that each dataset contains the expected fields and that dates and
numeric fields have been loaded using appropriate data types.

In [3]:
for name, df in datasets.items():
    print(f"\n{'=' * 70}")
    print(name.upper())
    print(f"{'=' * 70}")
    display(
        pd.DataFrame({
            "column": df.columns,
            "dtype": df.dtypes.astype(str).values,
        })
    )


CUSTOMERS


,column,dtype
0,customer_id,object
1,signup_date,datetime64[ns]
2,country,object
3,age,float64
4,segment,object
5,acquisition_channel,object
6,newsletter_opt_in,bool
7,favorite_color,object
8,zodiac_sign,object
9,preferred_language,object



PRODUCTS


,column,dtype
0,product_id,object
1,product_name,object
2,category,object
3,supplier_id,object
4,price_eur,float64
5,unit_cost_eur,float64
6,weight_g,int64
7,color,object
8,launch_date,datetime64[ns]



ORDERS


...

,column,dtype
0,order_id,object
1,product_id,object
2,quantity,int64
3,unit_price_eur,float64
4,discount_pct,int64



CAMPAIGNS


,column,dtype
0,campaign_id,object
1,campaign_name,object
2,channel,object
3,start_date,datetime64[ns]
4,end_date,datetime64[ns]
5,budget_eur,int64



SUPPORT_TICKETS


,column,dtype
0,ticket_id,object
1,customer_id,object
2,order_id,object
3,created_at,datetime64[ns]
4,topic,object
5,channel,object
6,agent_id,object
7,resolution_hours,float64
8,satisfaction_score,int64


Signup_date in the customers table had a date format inconsistency fix pushed to loading file

## 3. Missing values

Missing values are not automatically data errors. Some fields are
expected to be optional, so missingness will be interpreted according
to the meaning of each field.

In [4]:
for name, df in datasets.items():
    print(f"\n{name.upper()}")
    display(missing_summary(df))


CUSTOMERS


,missing_count,missing_pct
age,217,6.873614
country,63,1.995565
signup_date,0,0.000000
customer_id,0,0.000000
segment,0,0.000000
acquisition_channel,0,0.000000
newsletter_opt_in,0,0.000000
favorite_color,0,0.000000
zodiac_sign,0,0.000000
preferred_language,0,0.000000



PRODUCTS


,missing_count,missing_pct
category,4,2.666667
product_id,0,0.000000
product_name,0,0.000000
supplier_id,0,0.000000
price_eur,0,0.000000
unit_cost_eur,0,0.000000
weight_g,0,0.000000
color,0,0.000000
launch_date,0,0.000000



ORDERS


...

,missing_count,missing_pct
order_id,0,0.0
product_id,0,0.0
quantity,0,0.0
unit_price_eur,0,0.0
discount_pct,0,0.0



CAMPAIGNS


,missing_count,missing_pct
campaign_id,0,0.0
campaign_name,0,0.0
channel,0,0.0
start_date,0,0.0
end_date,0,0.0
budget_eur,0,0.0



SUPPORT_TICKETS


,missing_count,missing_pct
resolution_hours,77,5.0
ticket_id,0,0.0
customer_id,0,0.0
created_at,0,0.0
order_id,0,0.0
topic,0,0.0
channel,0,0.0
agent_id,0,0.0
satisfaction_score,0,0.0


84% of the orders are not associated with a campaign id


## 4. Primary-key integrity

Check whether fields expected to uniquely identify records actually do so.

In [5]:
primary_keys = {
    "customers": ["customer_id"],
    "products": ["product_id"],
    "orders": ["order_id"],
    "campaigns": ["campaign_id"],
    "support_tickets": ["ticket_id"],
}

primary_key_results = []

for name, key_columns in primary_keys.items():
    df = datasets[name]

    duplicate_count_value = df.duplicated(
        subset=key_columns,
        keep=False
    ).sum()

    primary_key_results.append({
        "dataset": name,
        "primary_key": ", ".join(key_columns),
        "duplicate_records": duplicate_count_value,
        "status": "PASS" if duplicate_count_value == 0 else "CHECK",
    })

primary_key_results = pd.DataFrame(primary_key_results)

display(primary_key_results)

,dataset,primary_key,duplicate_records,status
0,customers,customer_id,74,CHECK
1,products,product_id,0,PASS
2,orders,order_id,448,CHECK
3,campaigns,campaign_id,0,PASS
4,support_tickets,ticket_id,0,PASS


## 5. Referential integrity

Check that relationships between datasets are valid.

For example:

- every order belongs to a known customer
- every order item belongs to a known order
- every order item references a known product
- every attributed order references a known campaign
- every support ticket references a known customer

In [6]:
referential_checks = {
    "orders → customers": (
        orders,
        "customer_id",
        customers,
        "customer_id",
    ),
    "order_items → orders": (
        order_items,
        "order_id",
        orders,
        "order_id",
    ),
    "order_items → products": (
        order_items,
        "product_id",
        products,
        "product_id",
    ),
    "orders → campaigns": (
        orders[orders["campaign_id"].notna()],
        "campaign_id",
        campaigns,
        "campaign_id",
    ),
    "support → customers": (
        support,
        "customer_id",
        customers,
        "customer_id",
    ),
    "support → orders": (
        support[support["order_id"].notna()],
        "order_id",
        orders,
        "order_id",
    ),
}

results = []

for relationship, (
    child_df,
    child_column,
    parent_df,
    parent_column,
) in referential_checks.items():

    invalid_values = foreign_key_check(
        child_df,
        child_column,
        parent_df,
        parent_column,
    )

    results.append({
        "relationship": relationship,
        "invalid_reference_count": len(invalid_values),
        "status": "PASS" if len(invalid_values) == 0 else "CHECK",
    })

referential_results = pd.DataFrame(results)

display(referential_results)

,relationship,invalid_reference_count,status
0,orders → customers,0,PASS
1,order_items → orders,0,PASS
2,order_items → products,0,PASS
3,orders → campaigns,0,PASS
4,support → customers,0,PASS
5,support → orders,0,PASS


## 6. Date ranges

Inspect the temporal coverage of each dataset.

Dates outside the stated export period are not automatically errors,
because customers, products, campaigns, or support interactions may
have started before the analysis period.

In [7]:
date_columns = {
    "customers": "signup_date",
    "products": "launch_date",
    "orders": "order_date",
    "campaigns_start": ("campaigns", "start_date"),
    "campaigns_end": ("campaigns", "end_date"),
    "support_tickets": "created_at",
}

date_results = []

for name, column_info in date_columns.items():

    if isinstance(column_info, tuple):
        dataset_name, column = column_info
        df = datasets[dataset_name]
    else:
        dataset_name = name
        column = column_info
        df = datasets[dataset_name]

    date_results.append({
        "dataset": dataset_name,
        "column": column,
        "min_date": df[column].min(),
        "max_date": df[column].max(),
        "missing": df[column].isna().sum(),
    })

date_results = pd.DataFrame(date_results)

display(date_results)

,dataset,column,min_date,max_date,missing
0,customers,signup_date,2023-02-08,2026-11-05,0
1,products,launch_date,2022-01-15,2024-06-26,0
2,orders,order_date,2019-01-01,2026-06-30,0
3,campaigns,start_date,2024-09-01,2026-03-09,0
4,campaigns,end_date,2024-09-30,2026-04-09,0
5,support_tickets,created_at,2024-07-02,2026-07-10,0


## 7. Customer and order date consistency

Check whether any orders occur before the customer's recorded signup date.

In [9]:
order_customer = orders.merge(
    customers[["customer_id", "signup_date"]],
    on="customer_id",
    how="left",
)

orders_before_signup = order_customer[
    order_customer["order_date"] < order_customer["signup_date"]
]

print(
    f"Orders before recorded signup: "
    f"{len(orders_before_signup):,}"
)

if len(orders_before_signup) > 0:
    display(
        orders_before_signup[
            [
                "order_id",
                "customer_id",
                "order_date",
                "signup_date",
            ]
        ].head(20)
    )

Orders before recorded signup: 180


,order_id,customer_id,order_date,signup_date
0,O000351,CU00066,2019-01-01,2024-06-25
1,O000450,CU00081,2019-01-01,2025-01-07
2,O000632,CU00115,2019-01-01,2024-09-24
3,O000710,CU00126,2019-01-01,2024-06-26
4,O000961,CU00169,2019-01-01,2023-10-13
5,O001325,CU00225,2019-01-01,2025-03-18
6,O001384,CU00236,2019-01-01,2024-09-14
7,O001447,CU00243,2019-01-01,2025-01-23
8,O001598,CU00273,2019-01-01,2024-11-09
9,O001978,CU00346,2019-01-01,2023-08-12


## 8. Campaign validity

Check campaign dates, duration, and budget values.

In [11]:
campaign_checks = pd.DataFrame({
    "check": [
        "start_date > end_date",
        "missing campaign_id",
        "non-positive budget",
    ],
    "count": [
        (campaigns["start_date"] > campaigns["end_date"]).sum(),
        campaigns["campaign_id"].isna().sum(),
        (campaigns["budget_eur"] <= 0).sum(),
    ],
})

display(campaign_checks)

,check,count
0,start_date > end_date,0
1,missing campaign_id,0
2,non-positive budget,0


In [12]:
campaigns["duration_days"] = (
    campaigns["end_date"] - campaigns["start_date"]
).dt.days + 1

display(
    campaigns[
        [
            "campaign_id",
            "campaign_name",
            "channel",
            "start_date",
            "end_date",
            "duration_days",
            "budget_eur",
        ]
    ].head(20)
)

,campaign_id,campaign_name,channel,start_date,end_date,duration_days,budget_eur
0,C01,Back to Uni 2024,Instagram,2024-09-01,2024-09-30,30,22000
1,C02,Black Friday 2024,Google Ads,2024-11-22,2024-12-02,11,48000
2,C03,Xmas Gifts 2024,Newsletter,2024-12-03,2024-12-22,20,6000
3,C04,New Year Fitness,Facebook,2025-01-02,2025-01-31,30,18000
4,C05,Spring Home Refresh,Pinterest,2025-03-10,2025-04-10,32,15000
5,C06,Easter Family Deals,Newsletter,2025-04-07,2025-04-21,15,4500
6,C07,Mother's Day,Instagram,2025-04-28,2025-05-11,14,14000
7,C08,Loyalty Points x2,App Push,2025-05-15,2025-06-15,32,3000
8,C09,Summer Mega Sale,TikTok Influencer,2025-06-15,2025-08-31,78,185000
9,C10,Back to Uni 2025,Instagram,2025-09-01,2025-09-30,30,24000


## 9. Campaign attribution consistency

Investigate whether orders attributed to campaigns occurred during the
corresponding campaign period.

An order with a campaign_id outside the campaign's stated dates is not
automatically invalid. It may reflect attribution windows or delayed
conversion behavior, so these cases will be investigated before being
included or excluded from campaign analysis.

In [13]:
order_campaign = orders.merge(
    campaigns[
        [
            "campaign_id",
            "campaign_name",
            "channel",
            "start_date",
            "end_date",
            "budget_eur",
        ]
    ],
    on="campaign_id",
    how="left",
)

attributed_orders = order_campaign[
    order_campaign["campaign_id"].notna()
].copy()

outside_campaign_period = attributed_orders[
    (attributed_orders["order_date"] < attributed_orders["start_date"])
    |
    (attributed_orders["order_date"] > attributed_orders["end_date"])
]

print(
    f"Attributed orders: {len(attributed_orders):,}"
)

print(
    f"Attributed orders outside campaign period: "
    f"{len(outside_campaign_period):,}"
)

display(
    outside_campaign_period[
        [
            "order_id",
            "order_date",
            "campaign_id",
            "campaign_name",
            "start_date",
            "end_date",
        ]
    ].head(20)
)

Attributed orders: 2,412
Attributed orders outside campaign period: 10


,order_id,order_date,campaign_id,campaign_name,start_date,end_date
14,O004605,2019-01-01,C03,Xmas Gifts 2024,2024-12-03,2024-12-22
23,O009411,2019-01-01,C08,Loyalty Points x2,2025-05-15,2025-06-15
27,O014504,2019-01-01,C09,Summer Mega Sale,2025-06-15,2025-08-31
28,O014595,2019-01-01,C10,Back to Uni 2025,2025-09-01,2025-09-30
29,O014766,2019-01-01,C09,Summer Mega Sale,2025-06-15,2025-08-31
7923,O014587,2025-09-01,C09,Summer Mega Sale,2025-06-15,2025-08-31
7924,O014778,2025-09-01,C09,Summer Mega Sale,2025-06-15,2025-08-31
7925,O014915,2025-09-01,C09,Summer Mega Sale,2025-06-15,2025-08-31
7943,O014361,2025-09-02,C09,Summer Mega Sale,2025-06-15,2025-08-31
7944,O014801,2025-09-02,C09,Summer Mega Sale,2025-06-15,2025-08-31


## 10. Order status and revenue

Inspect order statuses and monetary values before defining the revenue
measure used in the business analysis.

In [14]:
print("Order status distribution:")
display(
    orders["status"]
    .value_counts(dropna=False)
    .rename_axis("status")
    .reset_index(name="orders")
)

Order status distribution:


,status,orders
0,completed,13063
1,returned,1629
2,cancelled,472


In [15]:
status_revenue = (
    orders.groupby("status", dropna=False)
    .agg(
        orders=("order_id", "count"),
        total_amount_eur=("total_amount", "sum"),
        average_amount_eur=("total_amount", "mean"),
    )
    .reset_index()
)

display(status_revenue)

,status,orders,total_amount_eur,average_amount_eur
0,cancelled,472,168605.70,357.215466
1,completed,13063,4268007.50,326.724910
2,returned,1629,897777.24,551.121694


In [16]:
print("Currency values:")
display(
    orders["currency"]
    .value_counts(dropna=False)
    .rename_axis("currency")
    .reset_index(name="orders")
)

Currency values:


,currency,orders
0,EUR,13342
1,EUR,623
2,€,620
3,eur,579


## 11. Order and order-item reconciliation

Recalculate order values from the item-level data and compare them
against the order-level `total_amount`.

The calculation uses:

quantity × unit price × (1 − discount)

In [17]:
order_items_check = order_items.copy()

order_items_check["calculated_value"] = (
    order_items_check["quantity"]
    * order_items_check["unit_price_eur"]
    * (1 - order_items_check["discount_pct"] / 100)
)

item_totals = (
    order_items_check
    .groupby("order_id")
    .agg(
        calculated_order_value=("calculated_value", "sum"),
        item_count=("product_id", "count"),
    )
    .reset_index()
)

reconciliation = orders.merge(
    item_totals,
    on="order_id",
    how="left",
)

reconciliation["difference"] = (
    reconciliation["total_amount"]
    - reconciliation["calculated_order_value"]
)

reconciliation["reconciles"] = (
    reconciliation["difference"].abs() <= 0.01
)

print(
    "Orders that reconcile:",
    reconciliation["reconciles"].sum(),
)

print(
    "Orders that do not reconcile:",
    (~reconciliation["reconciles"]).sum(),
)

display(
    reconciliation[
        ~reconciliation["reconciles"]
    ].head(20)
)

Orders that reconcile: 14899
Orders that do not reconcile: 265


,order_id,customer_id,order_date,campaign_id,payment_method,device,browser,status,shipping_country,total_amount,currency,calculated_order_value,item_count,difference,reconciles
6,O001384,CU00236,2019-01-01,NaN,Apple Pay,mobile,Samsung Internet,returned,Germany,355.63,EUR,-89.810,2,445.440,False
184,O000119,CU00020,2024-07-15,NaN,Credit Card,mobile,Safari,completed,Germany,238.07,EUR,-61.690,2,299.760,False
203,O003045,CU00544,2024-07-17,NaN,Apple Pay,mobile,Samsung Internet,completed,Netherlands,-1838.31,EUR,1838.310,1,-3676.620,False
261,O000949,CU00168,2024-07-22,NaN,PayPal,mobile,Chrome,completed,Germany,-46.23,EUR,46.230,1,-92.460,False
318,O010981,CU01994,2024-07-26,NaN,Klarna,mobile,Safari,completed,France,80.97,EUR,31.090,2,49.880,False
490,O001948,CU00343,2024-08-10,NaN,SEPA,mobile,Chrome,completed,Netherlands,144.46,EUR,120.620,4,23.840,False
558,O005344,CU00965,2024-08-16,NaN,SEPA,mobile,Chrome,completed,Germany,267.15,EUR,94.190,2,172.960,False
595,O011423,CU02068,2024-08-19,NaN,Credit Card,mobile,Firefox,completed,Netherlands,52.81,EUR,0.000,1,52.810,False
666,O004250,CU00771,2024-08-25,NaN,PayPal,desktop,Firefox,completed,Germany,-67.61,EUR,67.610,2,-135.220,False
802,O006127,CU01106,2024-09-05,NaN,SEPA,desktop,Firefox,completed,Germany,-1144.34,eur,1144.340,2,-2288.680,False


In [18]:
item_sanity = {
    "quantity <= 0": (order_items["quantity"] <= 0).sum(),
    "discount < 0": (order_items["discount_pct"] < 0).sum(),
    "discount > 100": (order_items["discount_pct"] > 100).sum(),
    "unit price < 0": (order_items["unit_price_eur"] < 0).sum(),
}

display(
    pd.Series(item_sanity, name="count")
    .to_frame()
)

,count
quantity <= 0,120
discount < 0,0
discount > 100,0
unit price < 0,0


In [19]:
product_sanity = {
    "price <= 0": (products["price_eur"] <= 0).sum(),
    "unit cost < 0": (products["unit_cost_eur"] < 0).sum(),
    "weight <= 0": (products["weight_g"] <= 0).sum(),
    "cost > price": (
        products["unit_cost_eur"] > products["price_eur"]
    ).sum(),
}

display(
    pd.Series(product_sanity, name="count")
    .to_frame()
)

,count
price <= 0,0
unit cost < 0,0
weight <= 0,0
cost > price,3


In [20]:
customer_sanity = {
    "age < 13": (customers["age"] < 13).sum(),
    "age > 100": (customers["age"] > 100).sum(),
}

display(
    pd.Series(customer_sanity, name="count")
    .to_frame()
)

,count
age < 13,13
age > 100,6


## Categorical columns useful for analysis


In [21]:
categorical_columns = [
    "country",
    "segment",
    "acquisition_channel",
    "newsletter_opt_in",
    "preferred_language",
]

for column in categorical_columns:
    print(f"\n{column}")
    display(
        customers[column]
        .value_counts(dropna=False)
        .head(20)
    )


country


country
Germany        1511
Austria         294
Netherlands     270
France          234
Switzerland     194
Poland          158
Deutschland      66
DE               66
NaN              63
germany          60
GER              59
FR               39
Holland          22
PL               22
Österreich       21
CH               18
NL               18
AT               15
austria          14
Schweiz          13
Name: count, dtype: int64


segment


segment
Student         1137
Professional    1005
Family           632
Senior           383
Name: count, dtype: int64


acquisition_channel


acquisition_channel
Paid Social       1207
Organic Search     743
Direct             512
Referral           428
Newsletter         267
Name: count, dtype: int64


newsletter_opt_in


newsletter_opt_in
False    1894
True     1263
Name: count, dtype: int64


preferred_language


preferred_language
de    2379
fr     208
en     197
nl     189
pl     184
Name: count, dtype: int64

## Support tickets with invalid rating

In [22]:
support_sanity = {
    "negative resolution hours": (
        support["resolution_hours"] < 0
    ).sum(),
    "satisfaction below 1": (
        support["satisfaction_score"] < 1
    ).sum(),
    "satisfaction above 5": (
        support["satisfaction_score"] > 5
    ).sum(),
}

display(
    pd.Series(support_sanity, name="count")
    .to_frame()
)

,count
negative resolution hours,0
satisfaction below 1,13
satisfaction above 5,10


## Highlights after initial audit

- **Duplicate customer IDs**: 74 records involved
- **Duplicate order IDs**: 448 records involved
- **Orders before customer signup**: 180
- **Orders attributed outside campaign dates**: 10
- **Orders failing item-total reconciliation**: 265
- **Cancelled orders**: 472
- **Returned orders**: 1,629
- **Missing campaign attribution**: 12,752 orders (84.1%)
- **Inconsistent country names**: Multiple variants
- **Missing customer ages**: 217 (6.87%)
- **Missing customer countries**: 63 (2.00%)
- **Missing payment methods**: 151 (1.00%)
- **Invalid satisfaction scores**: 23


## Resolution 

## orders from a campaign falling outside the campaign

In [24]:
display(
    outside_campaign_period[
        [
            "order_id",
            "order_date",
            "campaign_id",
            "campaign_name",
            "start_date",
            "end_date",
        ]
    ]
)

,order_id,order_date,campaign_id,campaign_name,start_date,end_date
14,O004605,2019-01-01,C03,Xmas Gifts 2024,2024-12-03,2024-12-22
23,O009411,2019-01-01,C08,Loyalty Points x2,2025-05-15,2025-06-15
27,O014504,2019-01-01,C09,Summer Mega Sale,2025-06-15,2025-08-31
28,O014595,2019-01-01,C10,Back to Uni 2025,2025-09-01,2025-09-30
29,O014766,2019-01-01,C09,Summer Mega Sale,2025-06-15,2025-08-31
7923,O014587,2025-09-01,C09,Summer Mega Sale,2025-06-15,2025-08-31
7924,O014778,2025-09-01,C09,Summer Mega Sale,2025-06-15,2025-08-31
7925,O014915,2025-09-01,C09,Summer Mega Sale,2025-06-15,2025-08-31
7943,O014361,2025-09-02,C09,Summer Mega Sale,2025-06-15,2025-08-31
7944,O014801,2025-09-02,C09,Summer Mega Sale,2025-06-15,2025-08-31


##  orders before signup

In [26]:
display(
    orders_before_signup.assign(
        days_before_signup=(
            orders_before_signup["signup_date"]
            - orders_before_signup["order_date"]
        ).dt.days
    )[
        ["order_id", "customer_id", "order_date",
         "signup_date", "days_before_signup"]
    ].sort_values(
        "days_before_signup",
        ascending=False,
    ).head(50)
)

,order_id,customer_id,order_date,signup_date,days_before_signup
16,O005236,CU00945,2019-01-01,2026-11-02,2862
21,O008686,CU01569,2019-01-01,2026-04-14,2660
28,O014595,CU02828,2019-01-01,2025-08-04,2407
29,O014766,CU02971,2019-01-01,2025-07-16,2388
27,O014504,CU02752,2019-01-01,2025-07-02,2374
23,O009411,CU01707,2019-01-01,2025-05-22,2333
10,O002844,CU00508,2019-01-01,2025-03-29,2279
5,O001325,CU00225,2019-01-01,2025-03-18,2268
7,O001447,CU00243,2019-01-01,2025-01-23,2214
20,O008146,CU01475,2019-01-01,2025-01-15,2206


## checking duplicates

In [27]:
for name, df, key in [
    ("customers", customers, "customer_id"),
    ("orders", orders, "order_id"),
]:
    duplicated = df[df.duplicated(subset=[key], keep=False)]

    print(f"\n{name.upper()}")
    print("Records with duplicate IDs:", len(duplicated))
    print("Exact duplicate rows:", df.duplicated().sum())

    conflicting_ids = (
        duplicated.groupby(key)
        .apply(
            lambda group: len(group.drop_duplicates()) > 1,
            include_groups=False,
        )
    )

    print("IDs with conflicting records:", conflicting_ids.sum())


CUSTOMERS
Records with duplicate IDs: 74
Exact duplicate rows: 37
IDs with conflicting records: 0

ORDERS
Records with duplicate IDs: 448
Exact duplicate rows: 224
IDs with conflicting records: 0


We dont have duplicates with conflicting values on the same keys the dupes can be dropped

## Order Values not matching with amounts


In [28]:
display(
    reconciliation.loc[
        ~reconciliation["reconciles"],
        [
            "order_id",
            "status",
            "total_amount",
            "calculated_order_value",
            "difference",
        ],
    ].sort_values(
        "difference",
        key=lambda s: s.abs(),
        ascending=False,
    ).head(20)
)

,order_id,status,total_amount,calculated_order_value,difference
10309,O004850,returned,-2769.46,2769.460,-5538.920
203,O003045,completed,-1838.31,1838.310,-3676.620
10362,O013741,completed,-1606.50,1606.500,-3213.000
802,O006127,completed,-1144.34,1144.340,-2288.680
5445,O000250,completed,-1080.84,1080.840,-2161.680
8643,O013886,returned,-1004.00,1004.000,-2008.000
11457,O009883,returned,3023.43,1185.120,1838.310
9487,O004858,returned,-863.95,863.950,-1727.900
9966,O013138,completed,1472.64,0.000,1472.640
13603,O012801,completed,1331.06,-141.580,1472.640


## Data Quality Issues and Analytical Decisions

| Issue | Decision |
|---|---|
| Duplicate customer and order records | Remove exact duplicates. No conflicting IDs were found. |
| Duplicate order-item rows | Retain, as identical rows may represent legitimate line items. |
| Orders dated `2019-01-01` | Treat as suspected placeholder dates and exclude from the July 2024–June 2026 reporting period. |
| Orders placed before customer signup | Retain orders with otherwise valid dates; flag the inconsistency. |
| Completed, returned, and cancelled orders | Use completed orders for realized revenue. Analyze returns and cancellations separately. |
| Negative order amounts | Retain and flag for interpretation; do not automatically change their values. |
| Order totals not matching item totals | Use recorded order totals for primary revenue calculations and document reconciliation limitations. |
| Orders attributed outside campaign dates | Retain recorded attribution and flag exceptions. |
| Missing `campaign_id` (84.1% of orders) | Treat as unattributed, not as evidence of no marketing exposure or a valid control group. |
| Inconsistent country names and codes | Standardize when performing geographic analysis. |
| Missing customer age, country, and payment method | Retain missing values; exclude only where the specific variable is required. |
| Invalid support satisfaction scores | Exclude invalid scores from satisfaction calculations. |
| Records outside the analysis period | Restrict primary reporting to July 2024–June 2026. Use earlier historical data only where relevant and reliable. |

